# Notebook C: extra runs (base learning-rate check: recommended; Q2 recovery: optional)

**Runtime: T4 GPU.** About 11 min per run.
- **Step 1 (recommended, about 25 min):** the brief requires hyperparameters tuned on validation data. The base experiments used learning rate 1e-5; this trains base generation 0 at 3e-6 and 3e-5 too and compares on `fsl_val_rl`. If 1e-5 is best or within noise, the existing base runs stand. Otherwise, report it as a limitation (rerunning everything would cost about 6 GPU hours).
- **Step 2 (optional, about 45 min): Q2, recovery.** Can a collapsed model recover when retrained on real data? It follows team guide section 3.2: T=2 generation 10 and T=1 generation 10 retrained on the real pool, plus Control A (generation 0 trained longer) and Control B (T=2 generation 10 trained further on its own synthetic pool).

## Setup 1: connect Google Drive
A pop-up asks for permission: pick your account → **Allow**. The last lines must say `collapse found: True`. If it says False, the shortcut from guide Part 3 is missing or has a different name.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
print('collapse found:', os.path.exists('/content/drive/MyDrive/collapse/gen0_pool/config.json'))
os.makedirs('/content/drive/MyDrive/nlp_p1/results', exist_ok=True)
os.makedirs('/content/drive/MyDrive/nlp_p1/collapse_ext', exist_ok=True)
!nvidia-smi --query-gpu=name --format=csv,noheader || echo "no GPU (fine for analysis, slower)"


## Setup 2: Python 3.10 with the project's versions (2–4 min)
The last line should show `0.4.26` and a device list.

In [ ]:
!pip -q install uv
!uv venv /content/py310 --python 3.10 -q --allow-existing
!uv pip install -q --python /content/py310/bin/python "jax[cuda12]==0.4.26" "jaxlib==0.4.26" "equinox==0.11.10" "optax==0.2.2" "nptyping==2.5.0" "numpy==1.26.4" "scipy==1.15.3" "h5py==3.16.0" "matplotlib==3.10.9" "tqdm==4.70.1"
!XLA_PYTHON_CLIENT_PREALLOCATE=false /content/py310/bin/python -c "import jax; print(jax.__version__, jax.devices())"


## Setup 3: the code (your branch)
If this fails with `Remote branch … not found`, push your branch first (guide Part 2).

In [ ]:
BRANCH = 'mivuyo/q1-induction-heads'   # your branch (it contains the extended code and interp/)
%cd /content
!rm -rf icl-dynamics && git clone -q -b $BRANCH https://github.com/Bigjay101/icl-dynamics.git
%cd /content/icl-dynamics
!git log --oneline -1

## Setup 4: links and settings (run this every time you reconnect)
This makes `collapse/`, `collapse_ext/` and `results/` inside the code folder point to your Drive, so every command in the guide works unchanged and every output lands in Drive.

In [ ]:
%cd /content/icl-dynamics
!ln -sfn /content/drive/MyDrive/collapse collapse
!ln -sfn /content/drive/MyDrive/nlp_p1/collapse_ext collapse_ext
!rm -rf results && ln -sfn /content/drive/MyDrive/nlp_p1/results results
import os
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['PY'] = '/content/py310/bin/python'
os.environ['TRAIN_ARGS'] = "--data_file omniglot_resnet18_randomized_order_s0.h5 --mixing_coeffs 1.0 --pt_burstiness 1 --train_context_len 2 --fs_relabel 5 --fs_relabel_split 8 2 0 --exemplar_split 1 4 0 --pe_names fsl_train fsl_val_rl fsl_train_valex fsl_test_class --pe_classes train train train test --pe_exemplars train train val train --pe_fs_relabel_scheme train val train train --pe_burstiness 1 1 1 1 --train_bs 32 --eval_iters 1000 --lr 0.00001 --d_model 64 --class_split 50 1473 100 --raw_name"
!ls -la | grep -E "collapse|results"


## Step 1: base learning-rate check

In [ ]:
%%bash
cd /content/icl-dynamics
mkdir -p results/runs/base_lr_sweep
for LR in 0.000003 0.00003; do
  RUN=lr_$LR; OUT=results/runs/base_lr_sweep
  if [ -f $OUT/$RUN/checkpoints/00001000000.eqx ]; then echo "$RUN already done"; continue; fi
  rm -rf $OUT/$RUN
  echo "=== base gen0 at $LR ($(date +%H:%M)) ==="
  $PY main.py $TRAIN_ARGS --lr $LR --train_iters 1000000 --eval_every 5000 --ckpt_every 100000 --init_seed 5 \
    --pool_size 1000000 --base_folder $OUT --run $RUN > $OUT/$RUN.log 2>&1 || { tail -n 20 $OUT/$RUN.log; exit 1; }
  grep "total training time" $OUT/$RUN.log
done

In [ ]:
import h5py, numpy as np, csv
runs = {'1e-5 (gen0_pool, used)': 'collapse/gen0_pool/log.h5',
        '3e-6': 'results/runs/base_lr_sweep/lr_0.000003/log.h5', '3e-5': 'results/runs/base_lr_sweep/lr_0.00003/log.h5'}
rows = []
for name, p in runs.items():
    with h5py.File(p) as f:
        loss = f['fsl_val_rl/loss'][-1]; acc = f['fsl_val_rl/in_context_acc'][-1]
        rows.append({'lr': name, 'val_loss': float(loss.mean()), 'val_loss_se': float(loss.std(ddof=1) / np.sqrt(len(loss))),
                     'val_in_context_acc': float(acc.mean())})
for r in rows: print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()})
with open('results/lr_sweep_base.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

## Step 2 (optional): Q2 recovery runs
Each run loads a model's final checkpoint and continues for 1M more sequences (to iteration 2,000,000) with a **fresh optimiser state** (team guide section 5).

In [ ]:
%%bash
cd /content/icl-dynamics
OUT=results/runs/q2; mkdir -p $OUT
run() {  # name  checkpoint_folder  pool
  NAME=$1; SRC=$2; POOL=$3
  if [ -f $OUT/$NAME/checkpoints/00002000000.eqx ]; then echo "$NAME already done"; return; fi
  rm -rf $OUT/$NAME
  echo "=== $NAME ($(date +%H:%M)) ==="
  $PY main.py $TRAIN_ARGS --train_iters 2000000 --eval_every 5000 --ckpt_every 100000 --init_seed 5 \
    --pool_file $POOL --load_from_ckpt $SRC/checkpoints/00001000000.eqx --load_from_ckpt_cfg $SRC/config.json \
    --no_load_opt_state --base_folder $OUT --run $NAME > $OUT/$NAME.log 2>&1 || { tail -n 20 $OUT/$NAME.log; exit 1; }
  grep "total training time" $OUT/$NAME.log
}
run recover_T2_gen10 collapse/sample_T2/gen10 collapse/pool_gen0.h5
run recover_T1_gen10 collapse/sample_T1/gen10 collapse/pool_gen0.h5
run controlA_gen0_longer collapse/gen0_pool collapse/pool_gen0.h5
run controlB_T2_gen10_own_pool collapse/sample_T2/gen10 collapse/sample_T2/pool_gen10.h5

### Q2 results: recovery speed and completeness, and whether the circuit re-forms

In [ ]:
!$PY find_phases.py results/runs/q2 --metric fsl_val_rl/in_context_acc --out results/q2_phases | grep -E "results/runs|->|10% / 50%|NOTE"
!for r in recover_T2_gen10 recover_T1_gen10 controlA_gen0_longer controlB_T2_gen10_own_pool; do $PY -m interp.find_heads_gen0 results/runs/q2/$r --out results/interp/q2_$r > /dev/null && echo "$r: heads -> results/interp/q2_$r/gen0_heads.json"; done

Note: in these runs the iteration counter starts at 1,000,000 (it continues from the checkpoint). So subtract 1,000,000 from the phase-timing iterations before comparing them with generation 0's 247,220 (50%) and 463,014 (90%).